# GRPO Loss

$$
\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \left[ \min \left( \frac{\pi_{\theta}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} | q, o_{i,<t})} \hat{A}_{i,t}, \text{clip} \left( \frac{\pi_{\theta}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} | q, o_{i,<t})}, 1-\epsilon, 1+\epsilon \right) \hat{A}_{i,t} \right) - \beta \mathbb{D}_{\text{KL}}[\pi_{\theta} || \pi_{\text{ref}}] \right] 
$$

$$
\mathbb{D}_{\text{KL}}[\pi_{\theta} || \pi_{\text{ref}}] = \frac{\pi_{\text{ref}}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta}(o_{i,t} | q, o_{i,<t})} - \log \frac{\pi_{\text{ref}}(o_{i,t} | q, o_{i,<t})}{\pi_{\theta}(o_{i,t} | q, o_{i,<t})} - 1
$$

$$
\hat{A}_{i,t} = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}
$$

In [2]:
import torch
import torch.nn.functional as F

In [3]:
def grpo_kl(pi_logprob, pi_ref_logprob):
    return pi_logprob.exp() - (pi_ref_logprob-pi_logprob) - 1

In [4]:
def grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob, advantage, input_len, len_oi):
    epsilon = 0.2
    beta = 0.01
    bs, seq_len = pi_logprob.shape
    len_oi = torch.tensor([len_oi] * bs, dtype = torch.long)
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1
    
    # GRPO Loss
    ratio = torch.exp(pi_logprob - pi_old_logprob)
    ratio_clip = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)
    advantage = advantage.unsqueeze(dim = 1) # [a, b ,c] -> [[a], [b], [c]]
    policy_gradient = torch.minimum(ratio * advantage , ratio_clip * advantage)
    kl = grpo_kl(pi_logprob, pi_ref_logprob)
    
    loss = (policy_gradient -  beta * kl) * mask
    loss = (-1 / bs ) * (1/len_oi.unsqueeze(dim = 1)) * loss  
    loss = loss.sum()

    return loss

In [ ]:
# 输出分布
pi_logits = torch.randn(3, 5, 32) # batch, seq_len, vocab_size
pi_ref_logits = torch.randn(3, 5, 32)
pi_old_logits = torch.randn(3, 5, 32)
# 获取log prob
pi_logprob = F.log_softmax(pi_logits, dim = -1)
print(pi_logprob[0, 1, 12:13])
print(f"****** pi_logprob shape is {pi_logprob.shape} ******")
pi_ref_logprob = F.log_softmax(pi_ref_logits, dim = -1)
pi_old_logprob = F.log_softmax(pi_old_logits, dim = -1)

# group data
token_ids = torch.tensor([[11, 12, 13, 14, 15], # 输入为11,12,13, 输出为:14, 15
                          [11, 12, 13, 15, 16],
                          [11, 12, 13, 16, 17],])
# 获取policy
pi_logprob = torch.gather(pi_logprob, dim=-1, index=token_ids.unsqueeze(-1)).squeeze(-1)
print(f"****** After gather pi_logprob shape is {pi_logprob.shape} ******")
print(pi_logprob[0, 1])
pi_ref_logprob = torch.gather(pi_ref_logprob, dim=-1, index=token_ids.unsqueeze(-1)).squeeze(-1)
pi_old_logprob = torch.gather(pi_old_logprob, dim=-1, index=token_ids.unsqueeze(-1)).squeeze(-1)
loss = grpo_loss(pi_logprob, pi_old_logprob, pi_ref_logprob, torch.tensor([-1, 2, 1]), 3, 2)
print(loss)


tensor([-3.3144])
****** pi_logprob shape is torch.Size([3, 5, 32]) ******
****** After gather pi_logprob shape is torch.Size([3, 5]) ******
tensor(-3.3144)
tensor(-0.3152)


# Trl implementation

ppo clip ratio
grpo clip ratio
trl "not" clip ratio, it haven't minibatch,  $exp(logprob - logprob.detach())$ always equal 1


In [26]:
policy = torch.tensor([0.5])
old_policy = torch.tensor([0.5])
print(f"policy shape is {policy.shape}")
ratio = policy/old_policy
print(ratio)

ratio = torch.exp( policy.log() - old_policy.log())
print(ratio)

policy shape is torch.Size([1])
tensor([1.])
tensor([1.])


In [27]:
gradient = -0.2
policy_gradient = - gradient * ( 1 / old_policy)
print(policy_gradient)

tensor([0.4000])
